In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from scipy import stats
import joblib

In [2]:
df = pd.read_csv(r'./Water Quality Prediction.csv')

# Select only the relevant features
features = df[['pH', 'Turbidity', 'TDS', 'Conductivity']]
target = df['Target']

In [3]:
Q1 = features.quantile(0.25)
Q3 = features.quantile(0.75)
IQR = Q3 - Q1
filter = (features > (Q1 - 1.5 * IQR)) & (features < (Q3 + 1.5 * IQR))
features_filtered = features[filter.all(axis=1)]
target_filtered = target[filter.all(axis=1)]

In [4]:
# Further outlier removal using Isolation Forest
iso_forest = IsolationForest(contamination=0.05)
outliers = iso_forest.fit_predict(features_filtered)
features_filtered = features_filtered[outliers == 1]
target_filtered = target_filtered[outliers == 1]

In [5]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(features_filtered, target_filtered, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [6]:
# Train the XGBoost model with additional parameters
model = xgb.XGBClassifier(
    use_label_encoder=False,
    eval_metric='logloss',
    learning_rate=0.05,  
    n_estimators=200,  
    max_depth=6,  
    subsample=0.8,  
    colsample_bytree=0.8,  
    gamma=0.1,  
    reg_alpha=0.01,  
    reg_lambda=0.01,  
    scale_pos_weight=1,  
    objective='binary:logistic'  
)

In [7]:
# Fit the model
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

c:\Users\Mr. Dhruhil\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [14:39:23] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Accuracy: 86.09%

Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.95      0.92     98510
           1       0.65      0.41      0.50     20271

    accuracy                           0.86    118781
   macro avg       0.77      0.68      0.71    118781
weighted avg       0.85      0.86      0.85    118781



In [8]:
# Save the trained model and scaler to disk as .pkl files
joblib.dump(model, 'xgb_best_model.pkl')  # Save the model
joblib.dump(scaler, 'scaler.pkl')  # Save the scaler

['scaler.pkl']